# STAT 764 · Meeting 19 — Same score, different models

**Tuesday, November 10**

In Meeting 18 four models nearly tied on the readmission patients. Today we ask what two of them learned:
logistic regression and the random forest, fitted on the same 15,000 patients. There are three questions anyone
asks of a fitted model, and each has a standard tool that answers it:

| the question | the standard tool that answers it |
|---|---|
| Which columns does the model rely on, across all patients? | **permutation importance** |
| How does the prediction move as one column changes? | **partial dependence**, with **ICE** (individual conditional expectation) curves for individual patients |
| Why did this one patient get this score? | **SHAP values** (SHapley Additive exPlanations) |

Thursday we break all three. Today, learn what each one says.

The page `interpretation.html` in this folder has the same ideas with pictures. Open it in a web browser.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
# Categories as text, counts as decimals: today's tools change one column's values, and decimals keep that exact.
X_all = enc[NUM + CAT].astype({**{c: str for c in CAT}, **{c: float for c in NUM}})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()

# Meeting 16's sample: a random 15,000 of the training patients, with all their stays.
pool = enc["patient_nbr"].iloc[train].unique()
chosen = np.random.default_rng(16).choice(pool, 15_000, replace=False)
rows = train[np.isin(enc["patient_nbr"].iloc[train], chosen)]
X_fit, y_fit = X_all.iloc[rows], y_all.iloc[rows].to_numpy()


def prep():
    """Meeting 9's preprocessing: fill and scale the counts, one-hot the categories."""
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
    ])


def make_pipe(model):
    """Preprocessing inside the pipeline, any model at the end."""
    return Pipeline([("prep", prep()), ("model", model)])


# The two models from Meeting 18's scoreboard, with the settings they earned (Meetings 9 and 16).
logistic = make_pipe(LogisticRegression(max_iter=2000)).fit(X_fit, y_fit)
forest = make_pipe(RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=5,
                                          n_jobs=-1, random_state=0)).fit(X_fit, y_fit)
print(f"test: {len(y_test):,} stays · came back {y_test.mean():.1%}")
print(f"AUC on test: logistic regression {roc_auc_score(y_test, logistic.predict_proba(X_test)[:, 1]):.4f}"
      f" · forest {roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]):.4f}")

## 1. Which columns does the model rely on? Permutation importance

Meeting 16 met this tool. Take the test stays and shuffle **one column**, so its values land on the wrong
patients, and leave everything else alone. If the model relied on that column, its AUC falls. Repeat five
times for each column and average the drop.

Two things to keep from Meeting 16: do it on **test** stays, so the drop is about new patients; and the
drop describes **this model**, not the hospital. A column the model ignores can still matter to patients.

In [ ]:
# Shuffle each column of the test stays 5 times and record the average fall in AUC. About 6 seconds.
imp = permutation_importance(logistic, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=764)
importance_logistic = pd.DataFrame({"AUC drop": imp.importances_mean, "spread over 5 shuffles": imp.importances_std},
                                   index=X_test.columns).sort_values("AUC drop", ascending=False)
print("logistic regression: the 8 columns whose shuffling costs the most AUC")
print(importance_logistic.head(8).round(4).to_string())

Two columns carry nearly all of it: **prior inpatient stays** and **where the patient was discharged to**.
Shuffling either one costs about 0.05 of AUC; everything else costs under 0.01.

## 2. How does the prediction move with one column? Partial dependence and ICE

Importance says a column matters. It does not say **how**. To see that, take some patients, set everyone's
prior stays to 0 and predict, then to 1, then 2, and so on, keeping each patient's other columns as they
are. Each patient gives one line: that is an **ICE curve**, short for **individual conditional expectation**. The
average of the lines is the **partial dependence** curve.

`curves` below does exactly that and returns one row per patient, one column per value.

In [ ]:
def curves(model, patients, column, values):
    """Each patient's predicted chance as `column` is set to each of `values`, everything else unchanged.

    Returns an array with one row per patient (an ICE curve) and one column per value; its column means are
    the partial dependence curve.
    """
    out = []
    for v in values:
        changed = patients.copy()
        changed[column] = v                              # everyone gets the same value of this one column
        out.append(model.predict_proba(changed)[:, 1])
    return np.array(out).T


def plot_curves(ice, values, title):
    """Thin lines: 40 patients' ICE curves. Thick line: the partial dependence, the average of all of them."""
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(values, ice[:40].T, color="0.6", lw=0.8)
    ax.plot(values, ice.mean(axis=0), color="black", lw=3, label="partial dependence (the average)")
    ax.set(xlabel="prior inpatient stays", ylabel="predicted chance", title=title, ylim=(0, 0.7))
    ax.legend(loc="upper left")
    plt.show()


ice_patients = X_test.sample(200, random_state=19)      # 200 test stays, the same ones all day
stays = np.arange(0, 8, dtype=float)                    # prior stays 0, 1, ..., 7
ice_logistic = curves(logistic, ice_patients, "number_inpatient", stays)
print("logistic regression, partial dependence at 0-7 prior stays:", " · ".join(f"{v:.3f}" for v in ice_logistic.mean(axis=0)))
plot_curves(ice_logistic, stays, "Logistic regression")

The average rises from about 10% at no prior stays to about 39% at seven. The thin lines show it is not
one story: at seven prior stays, the patients' own curves end anywhere from about 24% to 56%, because
their other columns differ.

## 3. Why did this patient get this score? SHAP values

Take one test stay that logistic regression scores at about 30%. Where did the extra 20 points over a
typical patient come from?

For a model that adds things up on the log-odds scale, as logistic regression does, there is an exact
answer. Each column contributes **its coefficient times (this patient's value − the average value)**.
Those contributions, added to the score of a patient with every column at its average, give this patient's
score exactly. They are the **SHAP values** for this model, short for **SHapley Additive exPlanations**. `explain_linear` computes them and adds up the
pieces that belong to each original column (all the one-hot columns of `age`, for example).

For a forest the same idea needs a package that is not in our environment (`shap`, with its TreeSHAP
algorithm); the page shows what it looks like.

In [ ]:
def explain_linear(pipe, patient):
    """Exact SHAP values of a fitted logistic-regression pipeline for one patient (a one-row DataFrame).

    Returns (contributions, base): contributions in log-odds, one per original column, largest first; base is
    the log-odds of a patient with every column at its training average. base + contributions.sum() is this
    patient's log-odds.
    """
    pre, lr = pipe[:-1], pipe[-1]
    dense = lambda z: z.toarray() if hasattr(z, "toarray") else np.asarray(z)
    average = dense(pre.transform(X_fit)).mean(axis=0)              # the average patient, after preprocessing
    pieces = lr.coef_[0] * (dense(pre.transform(patient))[0] - average)
    names = pre.get_feature_names_out()                              # e.g. num__number_inpatient, cat__age_[80-90)
    column = [n.split("__", 1)[1] if n.startswith("num__") else
              max((c for c in CAT if n.split("__", 1)[1].startswith(c + "_")), key=len) for n in names]
    contributions = pd.Series(pieces, index=column).groupby(level=0).sum()
    base = lr.intercept_[0] + lr.coef_[0] @ average
    return contributions.sort_values(key=abs, ascending=False), base


p_logistic = logistic.predict_proba(X_test)[:, 1]
who = np.flatnonzero((p_logistic > 0.29) & (p_logistic < 0.31) & (X_test["number_inpatient"].to_numpy() <= 3))[0]
patient = X_test.iloc[[who]]
contributions, base = explain_linear(logistic, patient)
print(f"test stay {who}: predicted chance {p_logistic[who]:.1%}")
print(f"a patient with every column at its average: log-odds {base:.3f}, a chance of {expit(base):.1%}")
print("\nthe largest contributions, in log-odds (+ pushes the chance up):")
print(contributions.head(8).round(3).to_string())
print(f"\naverage {base:.3f} + all contributions {contributions.sum():+.3f} = {base + contributions.sum():.3f}"
      f" log-odds = {expit(base + contributions.sum()):.1%}")
print("\nthis patient:", " · ".join(f"{c} {patient[c].iloc[0]}" for c in ["discharge_disposition_id", "number_inpatient",
                                                                     "age", "medical_specialty", "time_in_hospital"]))

Discharge code 5, a transfer to another inpatient care facility, adds the most; two prior stays add about
half as much. Converted back to chances, the average patient's 10.4% becomes this patient's 29.9%.

## Studio

**Everything you need is in this notebook.** Sections 1–3 above show each tool on logistic regression. The
studio is an investigation: two models with the same AUC, and the same two important columns, can still give the
same patient very different chances. Find where, and use the tools and the real outcomes to work out why.

| | what you practice | about |
|---|---|---|
| **Question 1** | comparing what two models rely on | 10 minutes |
| **Question 2** | finding the patients two models disagree about | 15 minutes |
| **Question 3** | judging which model is closer, and explaining why | 20 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24
hours after class.

**How to work.**

1. If you opened this notebook fresh, run everything above first (**Run All**). The questions use `logistic`,
   `forest`, `X_test`, `y_test`, `X_fit`, `importance_logistic`, `curves`, `plot_curves`, `stays` and
   `explain_linear`.
2. Before each question, **write your guess** as a comment in its cell, and keep it when the answer
   disagrees. The Compare uses it.
3. Each question ends with sentences to answer. Write them in a comment or a markdown cell under its output.
4. Save and upload to Canvas under "Studio notebook — Day 19" by **Wednesday, November 11, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `logistic`, `curves` or `explain_linear` | Run the notebook from the top (Run All). |
| The forest's permutation importance seems stuck | It takes about 20 seconds: 20 columns, 5 shuffles each. |
| Every AUC drop is 0 | You passed the training rows. Use `X_test` and `y_test`. |
| The biggest differences come out smallest | `np.argsort` sorts smallest first: sort `-np.abs(gap)` instead. |
| `curves` or `explain_linear` fails on one stay | Pass a one-row DataFrame, `X_test.iloc[[i]]`, with double brackets. |

### 1. Do the two models rely on the same columns?

- **(a)** Run `permutation_importance` on the **forest**, as Section 1 did for logistic regression (same test
  stays, `scoring="roc_auc"`, `n_repeats=5`, `random_state=764`).
- **(b)** One table: the forest's 8 most important columns, with logistic regression's drop beside each.

**Answer in a sentence or two:** do the two models rely on the same columns? Then guess before question 2: if
they rely on the same two columns, can they still give one patient very different chances?

### 2. Where do the two models disagree?

- **(a)** Each model's chance for every test stay. How many stays get chances more than 10 points apart?
- **(b)** The 5 stays with the largest difference: for each, the number of prior stays, the discharge type
  (`discharge_disposition_id`), both chances, and whether the patient came back.
- **(c)** Among **all** the stays more than 10 points apart: their average number of prior stays, beside the
  average over all test stays, and the share where logistic regression gives the higher chance.

**Answer in two or three sentences:** what do the stays the models disagree about have in common, and which
shape from Section 2 explains the disagreement?

### 3. Which model is closer, and can the tools tell you why?

- **(a)** Among the stays more than 10 points apart: the share that actually came back, beside each model's
  average chance for them.
- **(b)** How many of the training stays (`X_fit`) had 5 or more prior stays? 10 or more?
- **(c)** The stay with the largest disagreement: its SHAP values from `explain_linear`, and both models' ICE
  curves for prior stays. `curves` works on a single patient, `X_test.iloc[[i]]`.

**Answer in three or four sentences:** for these patients, which model is closer to what happened? Why might each
model behave as it does there, given how many training stays look like them? One model scores this patient far
higher than the other: what would you tell the nurse about either number?

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class (Thursday, November 12), built from the notebooks you upload.
So leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 2: your five stays, and what they have in common.
3. Question 3: the share who came back, beside each model's average chance.

## Exit ticket

> Both models rank prior stays among the two most important columns. In two or three sentences: does that
> mean preventing a patient's earlier stays would lower their chance of readmission? Why or why not?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook —
Day 19". Your two studio points need this answered: a blank exit ticket earns zero. It is not graded for
being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ESL §10.13.1–10.13.2** — this week's reading: relative importance and partial dependence, the source of
Sections 1 and 2.

📗 **ESL §15.3.2** — variable importance for random forests, and its limits.

📗 **The `shap` package** (`pip install shap` in a separate environment) — TreeSHAP computes exact SHAP values
for forests and boosted models; `shap.Explainer(forest[-1])` on the preprocessed rows is the place to start.

🎓 **Meeting 20 (Thursday)** — breaking all three tools: columns that carry the same information,
predictions for patients who do not exist, and why an explanation is not a cause.